# Фінальний проєкт: infectious_cases (PostgreSQL)

Дані: статистика інфекційних захворювань за країнами та роками. СУБД: локальний PostgreSQL 16 (pgAdmin), база `fp`, схема `pandemic`.

## Завдання 1. Завантаження даних

In [2]:
%pip install -q psycopg2-binary sqlalchemy pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import os
import getpass
import pandas as pd
from sqlalchemy import create_engine, text

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

# Пароль не зберігається у файлі: береться зі змінної середовища або вводиться вручну.
DB_URL = os.getenv("FP_DB_URL")
if not DB_URL:
    DB_URL = f"postgresql+psycopg2://postgres:{getpass.getpass('Пароль користувача postgres: ')}@localhost:5432/fp"

engine = create_engine(DB_URL, future=True)

with engine.connect() as conn:
    print(conn.execute(text("SELECT version()")).scalar_one())


def run(sql_text):
    """Виконує SQL (можна кілька команд). Якщо остання повертає рядки, віддає DataFrame."""
    with engine.begin() as conn:
        res = conn.exec_driver_sql(sql_text.replace("%", "%%"))
        if res.returns_rows:
            return pd.DataFrame(res.fetchall(), columns=list(res.keys()))

PostgreSQL 16.15, compiled by Visual C++ build 1944, 64-bit


Створюємо схему `pandemic` і робимо її схемою за замовчуванням для всіх наступних запитів.

In [4]:
run(r"""
CREATE SCHEMA IF NOT EXISTS pandemic;
SET search_path TO pandemic;
""")

In [5]:
# Схему за замовчуванням задаємо на рівні бази, щоб вона діяла для кожного підключення з пулу
run("ALTER DATABASE " + engine.url.database + " SET search_path TO pandemic, public")
engine.dispose()
run("SHOW search_path")

,search_path
0,"pandemic, public"


Імпорт `infectious_cases.csv` у таблицю `infectious_cases` у схемі `pandemic`. Усі показники беремо як TEXT: у файлі є порожні та нечислові значення, тому спершу завантажуємо дані як є, а типи приводимо в запитах.

In [6]:
CSV_PATH = os.getenv("INFECTIOUS_CSV") or r"C:\Users\PC\Downloads\infectious_cases.csv"

df = pd.read_csv(CSV_PATH, dtype=str, keep_default_na=False)
df.columns = [c.strip().lower() for c in df.columns]
df["year"] = pd.to_numeric(df["year"], errors="coerce").astype("Int64")
df = df.replace("", None)

df.to_sql("infectious_cases", engine, schema="pandemic", if_exists="replace", index=False, chunksize=10_000)
print(CSV_PATH, df.shape)

C:\Users\PC\Downloads\infectious_cases.csv (10521, 12)


Перегляд структури та вмісту даних.

In [7]:
run(r"""
SELECT * FROM pandemic.infectious_cases LIMIT 10;
""")

,entity,code,year,number_yaws,polio_cases,cases_guinea_worm,number_rabies,number_malaria,number_hiv,number_tuberculosis,number_smallpox,number_cholera_cases
0,Afghanistan,AFG,1980,None,6160,0,None,None,None,None,None,None
1,Afghanistan,AFG,1981,None,5859,0,None,None,None,None,None,None
2,Afghanistan,AFG,1982,None,9730,0,None,None,None,None,None,None
3,Afghanistan,AFG,1983,None,13937,0,None,None,None,None,None,None
4,Afghanistan,AFG,1984,None,3864,0,None,None,None,None,None,None
5,Afghanistan,AFG,1985,None,13867,0,None,None,None,None,None,None
6,Afghanistan,AFG,1986,None,12901,0,None,None,None,None,None,None
7,Afghanistan,AFG,1987,None,4396,0,None,None,None,None,None,None
8,Afghanistan,AFG,1988,None,2149,0,None,None,None,None,None,None
9,Afghanistan,AFG,1989,None,385,0,None,None,None,None,None,None


In [8]:
run(r"""
SELECT COUNT(*) AS n_rows, COUNT(DISTINCT (entity, code)) AS n_entities FROM pandemic.infectious_cases;
""")

,n_rows,n_entities
0,10521,245


In [9]:
run(r"""
SELECT column_name, data_type
FROM information_schema.columns
WHERE table_schema = 'pandemic' AND table_name = 'infectious_cases'
ORDER BY ordinal_position;
""")

,column_name,data_type
0,entity,text
1,code,text
2,year,bigint
3,number_yaws,text
4,polio_cases,text
5,cases_guinea_worm,text
6,number_rabies,text
7,number_malaria,text
8,number_hiv,text
9,number_tuberculosis,text


## Завдання 2. Нормалізація до 3НФ

Створюємо дві таблиці: `entities` з унікальними комбінаціями Entity і Code та `infectious_cases_norm` з показниками, де замість повторюваних Entity і Code стоїть `entity_id`.

In [10]:
run(r"""
DROP TABLE IF EXISTS pandemic.infectious_cases_norm;
DROP TABLE IF EXISTS pandemic.entities;

CREATE TABLE pandemic.entities (
    id     SERIAL PRIMARY KEY,
    entity TEXT NOT NULL,
    code   TEXT,
    UNIQUE (entity, code)
);

INSERT INTO pandemic.entities (entity, code)
SELECT DISTINCT entity, code
FROM pandemic.infectious_cases
ORDER BY entity;
""")

In [11]:
run(r"""
CREATE TABLE pandemic.infectious_cases_norm (
    id                   SERIAL PRIMARY KEY,
    entity_id            INT NOT NULL REFERENCES pandemic.entities (id),
    year                 INT,
    number_yaws          TEXT,
    polio_cases          TEXT,
    cases_guinea_worm    TEXT,
    number_rabies        TEXT,
    number_malaria       TEXT,
    number_hiv           TEXT,
    number_tuberculosis  TEXT,
    number_smallpox      TEXT,
    number_cholera_cases TEXT
);

INSERT INTO pandemic.infectious_cases_norm (
    entity_id, year, number_yaws, polio_cases, cases_guinea_worm,
    number_rabies, number_malaria, number_hiv, number_tuberculosis,
    number_smallpox, number_cholera_cases
)
SELECT
    e.id, ic.year, ic.number_yaws, ic.polio_cases, ic.cases_guinea_worm,
    ic.number_rabies, ic.number_malaria, ic.number_hiv, ic.number_tuberculosis,
    ic.number_smallpox, ic.number_cholera_cases
FROM pandemic.infectious_cases ic
JOIN pandemic.entities e
  ON e.entity = ic.entity
 AND e.code IS NOT DISTINCT FROM ic.code;
""")

Перевірка: кількість рядків у нормалізованій таблиці має збігатися з сирою.

In [12]:
run(r"""
SELECT
    (SELECT COUNT(*) FROM pandemic.infectious_cases)      AS raw_rows,
    (SELECT COUNT(*) FROM pandemic.infectious_cases_norm) AS norm_rows,
    (SELECT COUNT(*) FROM pandemic.entities)              AS entities_rows;
""")

,raw_rows,norm_rows,entities_rows
0,10521,10521,245


In [13]:
# контроль: обидві таблиці існують і рядків стільки ж
chk = run("""
SELECT (SELECT COUNT(*) FROM pandemic.infectious_cases) AS raw_rows,
       (SELECT COUNT(*) FROM pandemic.infectious_cases_norm) AS norm_rows
""")
assert chk.loc[0, "raw_rows"] == chk.loc[0, "norm_rows"], "кількість рядків не збігається"
print("OK: нормалізація без втрат рядків")

OK: нормалізація без втрат рядків


In [14]:
run(r"""
SELECT e.entity, e.code, n.year, n.number_rabies, n.number_malaria
FROM pandemic.infectious_cases_norm n
JOIN pandemic.entities e ON e.id = n.entity_id
ORDER BY n.id
LIMIT 10;
""")

,entity,code,year,number_rabies,number_malaria
0,Afghanistan,AFG,1990,2.145772,1219845.8
1,Afghanistan,AFG,1991,2.3209748,1132781.5
2,Afghanistan,AFG,1995,3.126361,725758.3
3,Afghanistan,AFG,1997,3.446122,605853.06
4,Afghanistan,AFG,1998,3.5570247,547045.4
5,Afghanistan,AFG,1999,3.607037,502898.88
6,Afghanistan,AFG,2000,3.6362348,491614.72
7,Afghanistan,AFG,2001,3.7118323,511626.2
8,Afghanistan,AFG,2002,3.841707,546030.6
9,Afghanistan,AFG,2003,3.914814,574740.3


## Завдання 3. Аналіз Number_rabies

Для кожної унікальної комбінації Entity і Code: середнє, мінімум, максимум і сума. Беремо лише числові значення (порожні та нечислові відсікає регулярний вираз). Сортуємо за середнім за спаданням, виводимо 10 рядків.

In [15]:
run(r"""
SELECT
    e.id,
    e.entity,
    e.code,
    AVG(n.number_rabies::NUMERIC) AS avg_rabies,
    MIN(n.number_rabies::NUMERIC) AS min_rabies,
    MAX(n.number_rabies::NUMERIC) AS max_rabies,
    SUM(n.number_rabies::NUMERIC) AS sum_rabies
FROM pandemic.infectious_cases_norm n
JOIN pandemic.entities e ON e.id = n.entity_id
WHERE n.number_rabies ~ '^-?[0-9]+(\.[0-9]+)?$'
GROUP BY e.id, e.entity, e.code
ORDER BY avg_rabies DESC
LIMIT 10;
""")

,id,entity,code,avg_rabies,min_rabies,max_rabies,sum_rabies
0,241,World,OWID_WRL,20192.370366666667,14075.508,24744.658,605771.111
1,121,Lower Middle Income (WB),NaN,15193.959383333333,10202.528,19182.795,455818.7815
2,204,South Asia (WB),NaN,11729.888753333333,7271.2754,15361.878,351896.6626
3,201,South-East Asia Region (WHO),NaN,11424.326893333333,6806.0127,15641.958,342729.8068
4,77,G20,NaN,10189.0460033333333333,6339.0796,13164.881,305671.3801
5,97,India,IND,8599.1732800000000000,5425.8726,11121.142,257975.1984
6,209,Sub-Saharan Africa (WB),NaN,4748.2680900000000000,4356.8184,5017.977,142448.0427
7,3,African Region (WHO),NaN,4649.7023766666666667,4283.073,4917.784,139491.0713
8,120,Low Income (WB),NaN,3568.1054500000000000,3014.617,3873.533,107043.1635
9,58,East Asia & Pacific (WB),NaN,3427.4611333333333333,2009.9395,4591.0107,102823.8340


## Завдання 4. Різниця в роках (вбудовані функції)

`MAKE_DATE` будує 1 січня відповідного року, `CURRENT_DATE` дає поточну дату, `AGE` разом з `EXTRACT(YEAR ...)` дає повну різницю в роках.

In [16]:
run(r"""
SELECT
    n.id,
    e.entity,
    n.year,
    MAKE_DATE(n.year, 1, 1)                                             AS year_start,
    CURRENT_DATE                                                        AS today,
    EXTRACT(YEAR FROM AGE(CURRENT_DATE, MAKE_DATE(n.year, 1, 1)))::INT  AS year_diff
FROM pandemic.infectious_cases_norm n
JOIN pandemic.entities e ON e.id = n.entity_id
WHERE n.year IS NOT NULL
ORDER BY n.id
LIMIT 20;
""")

,id,entity,year,year_start,today,year_diff
0,1,Afghanistan,1990,1990-01-01,2026-10-13,36
1,2,Afghanistan,1991,1991-01-01,2026-10-13,35
2,3,Afghanistan,1995,1995-01-01,2026-10-13,31
3,4,Afghanistan,1997,1997-01-01,2026-10-13,29
4,5,Afghanistan,1998,1998-01-01,2026-10-13,28
5,6,Afghanistan,1999,1999-01-01,2026-10-13,27
6,7,Afghanistan,2000,2000-01-01,2026-10-13,26
7,8,Afghanistan,2001,2001-01-01,2026-10-13,25
8,9,Afghanistan,2002,2002-01-01,2026-10-13,24
9,10,Afghanistan,2003,2003-01-01,2026-10-13,23


## Завдання 5. Власні функції

**Функція 1** `year_diff(p_year)` приймає рік і повертає різницю в роках між поточною датою та 1 січня цього року.

**Функція 2** `avg_cases_per_period(p_cases, p_divisor)` повертає середню кількість захворювань за період: кількість за рік поділена на дільник (12 для місяця, 4 для кварталу, 2 для півріччя). `NULLIF` захищає від ділення на нуль.

In [17]:
run(r"""
CREATE OR REPLACE FUNCTION pandemic.year_diff(p_year INT)
RETURNS INT
LANGUAGE SQL
STABLE
AS $$
    SELECT EXTRACT(YEAR FROM AGE(CURRENT_DATE, MAKE_DATE(p_year, 1, 1)))::INT;
$$;
""")

In [18]:
run(r"""
CREATE OR REPLACE FUNCTION pandemic.avg_cases_per_period(p_cases NUMERIC, p_divisor NUMERIC)
RETURNS NUMERIC
LANGUAGE SQL
IMMUTABLE
AS $$
    SELECT p_cases / NULLIF(p_divisor, 0);
$$;
""")

Перевірка функцій на простих значеннях: `year_diff(1996)` і `avg_cases_per_period(120, 12 / 4 / 2)`.

In [19]:
run(r"""
SELECT
    pandemic.year_diff(1996)                         AS years_since_1996,
    pandemic.avg_cases_per_period(120, 12)           AS per_month,
    pandemic.avg_cases_per_period(120, 4)            AS per_quarter,
    pandemic.avg_cases_per_period(120, 2)            AS per_half_year;
""")

,years_since_1996,per_month,per_quarter,per_half_year
0,30,10.0000000000000000,30.0000000000000000,60.0000000000000000


Обидві функції в одному запиті, лише для записів із числовим значенням `Number_rabies`.

In [20]:
run(r"""
SELECT
    e.entity,
    n.year,
    pandemic.year_diff(n.year)                                              AS years_ago,
    n.number_rabies::NUMERIC                                                AS rabies_per_year,
    ROUND(pandemic.avg_cases_per_period(n.number_rabies::NUMERIC, 12), 2)   AS avg_per_month,
    ROUND(pandemic.avg_cases_per_period(n.number_rabies::NUMERIC, 4), 2)    AS avg_per_quarter,
    ROUND(pandemic.avg_cases_per_period(n.number_rabies::NUMERIC, 2), 2)    AS avg_per_half_year
FROM pandemic.infectious_cases_norm n
JOIN pandemic.entities e ON e.id = n.entity_id
WHERE n.number_rabies ~ '^-?[0-9]+(\.[0-9]+)?$'
ORDER BY e.entity, n.year
LIMIT 20;
""")

,entity,year,years_ago,rabies_per_year,avg_per_month,avg_per_quarter,avg_per_half_year
0,Afghanistan,1990,36,2.145772,0.18,0.54,1.07
1,Afghanistan,1991,35,2.3209748,0.19,0.58,1.16
2,Afghanistan,1992,34,2.6180408,0.22,0.65,1.31
3,Afghanistan,1993,33,2.8531172,0.24,0.71,1.43
4,Afghanistan,1994,32,2.9899871,0.25,0.75,1.49
5,Afghanistan,1995,31,3.126361,0.26,0.78,1.56
6,Afghanistan,1996,30,3.281437,0.27,0.82,1.64
7,Afghanistan,1997,29,3.446122,0.29,0.86,1.72
8,Afghanistan,1998,28,3.5570247,0.30,0.89,1.78
9,Afghanistan,1999,27,3.607037,0.30,0.90,1.80


## Підсумок

- Створено схему `pandemic`, дані з `infectious_cases.csv` імпортовано.
- Таблицю нормалізовано до 3НФ: `entities` (унікальні Entity і Code) та `infectious_cases_norm` (показники з `entity_id`), кількість рядків збігається з сирою таблицею.
- Для `Number_rabies` пораховано середнє, мінімум, максимум і суму, топ-10 за середнім.
- Різницю в роках побудовано вбудованими функціями `MAKE_DATE`, `CURRENT_DATE`, `AGE`.
- Створено дві функції (`year_diff`, `avg_cases_per_period`) і використано в запитах з дільниками 12, 4 і 2.